# Milestone 5

Master the competition metric, optimise inference, and combine models for maximum performance. Introduce TTA.

Suggested Readings:

PyTorch – Softmax Function
Hugging Face – Transformer
Ensemble Learning

---
---

In [1]:
import pandas as pd
import numpy as np

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

In [2]:
train

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A
...,...,...,...,...,...,...,...,...
1995,1996,What is the piezoelectric strain coefficient f...,d = 1.9·10‑12 m/V,d = 3.1·10‑12 m/V,d = 4.2·10‑12 m/V,d = 2.5·10‑12 m/V,d = 5.8·10‑12 m/V,B
1996,1997,Identify the correct statement: What is the sy...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,E
1997,1998,Determine the correct option: What does Earnsh...,A collection of point charges can be maintaine...,A collection of point charges can be maintaine...,A collection of point charges can be maintaine...,A collection of point charges cannot be mainta...,A collection of point charges can be maintaine...,D
1998,1999,Identify the correct statement: What is the re...,The atmosphere is a mechanism that is only inf...,"The atmosphere possesses both chaos and order,...",The atmosphere is a structure that is only inf...,The atmosphere is a completely chaotic mechani...,The atmosphere is a completely ordered structu...,B


Setup (Run Before Attempting Questions) : 

Use the following two models:

DeBERTa: microsoft/deberta-v3-small

RoBERTa: roberta-base

Label Mapping
The models output logits for five labels corresponding to the answer options:

Label ID         Option

0                       A

1                       B

2                       C

3                       D

4                       E

In [3]:
!pip -q install transformers datasets accelerate sentencepiece

import torch
import pandas as pd
from transformers import (AutoTokenizer,AutoModelForSequenceClassification)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


DEBERTA_CHECKPOINT = "microsoft/deberta-v3-small"
ROBERTA_CHECKPOINT = "roberta-base"

deberta_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_CHECKPOINT)
roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_CHECKPOINT)

deberta_model = AutoModelForSequenceClassification.from_pretrained(DEBERTA_CHECKPOINT, num_labels=5).to(device)
roberta_model = AutoModelForSequenceClassification.from_pretrained(ROBERTA_CHECKPOINT, num_labels=5).to(device)

deberta_model.eval()
roberta_model.eval()

print("Models loaded successfully!")


id2label = {0: "A",1: "B",2: "C",3: "D",4: "E"}
label2id = {v: k for k, v in id2label.items()}

print("setup complete!")


/home/deeepak/iitm_project/smart-mcq-solver-dlgenai-2026/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


Loading weights: 100%|██████████| 102/102 [00:00<00:00, 23289.00it/s]
[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING  

Models loaded successfully!
setup complete!


### Question 1:

Load the fine-tuned DeBERTa and RoBERTa models.

For the prompt at row index 25, perform inference using each model independently and apply Softmax to obtain class probabilities.

Question 1:

Which answer option receives the highest probability from the DeBERTa model, and what is that probability?
(answer format : eg - A, probability of A)

In [4]:
row = test.iloc[25]
text = row["prompt"]

inputs_deb = deberta_tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
with torch.no_grad():
    out_deb = deberta_model(**inputs_deb)
    deberta_probs = torch.softmax(out_deb.logits, dim=-1).cpu().numpy()[0]

# RoBERTa
inputs_rob = roberta_tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
with torch.no_grad():
    out_rob = roberta_model(**inputs_rob)
    roberta_probs = torch.softmax(out_rob.logits, dim=-1).cpu().numpy()[0]

labels = ["A", "B", "C", "D", "E"]
pred_idx = deberta_probs.argmax()
print(f"{labels[pred_idx]}, {deberta_probs[pred_idx]:.6f}")

A, 0.252686


### Question 2:

Using the same sample (row index 25), average the class probabilities from both models.

Average Probability = [P(DeBERTa) + P(RoBERTa)]/2

Question 2:

Which answer option receives the highest averaged probability after simple probability ensembling?

In [5]:
avg_probs = (deberta_probs.astype(float) + roberta_probs.astype(float)) / 2
pred_idx = avg_probs.argmax()
print(f"{labels[pred_idx]}, {avg_probs[pred_idx]:.6f}")

B, 0.219909


### Question 3:

Apply weighted probability averaging after Softmax using the following weights:

DeBERTa: 0.70

RoBERTa: 0.30

Compute:

P(final) = [0.7 × P(DeBERTa)] + [0.3 × P(RoBERTa)]

Question 3:

Which answer option is ranked first after weighted ensembling?

In [6]:
final_probs = 0.7 * deberta_probs.astype(float) + 0.3 * roberta_probs.astype(float)
pred_idx = final_probs.argmax()
print(f"{labels[pred_idx]}, {final_probs[pred_idx]:.6f}")

A, 0.227043


### Question 4:

Using the weighted ensemble probabilities from Q3, rank all five answer options.

Write the final prediction exactly in Kaggle submission format.

Question 4:

What is the Top-3 prediction string for row index 25?

Example : C A E



In [7]:
order = final_probs.argsort()[::-1]
top3 = [labels[i] for i in order[:3]]
top3_str = " ".join(top3)
print(top3_str)

A B E


### Question 5:

Run the weighted ensemble pipeline on every row of test.csv.

Save the predictions in a file named submission.csv using the required Kaggle format:

id,prediction

where the prediction column contains the Top-3 ranked options separated by spaces.

Question 5:

Exactly how many prediction rows are present in the generated file (excluding the header)?



In [8]:
submission_data = []

for idx, row in test.iterrows():
    text = row["prompt"]
    
    # DeBERTa inference
    inputs_deb = deberta_tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
    with torch.no_grad():
        out_deb = deberta_model(**inputs_deb)
        deb_probs = torch.softmax(out_deb.logits, dim=-1).cpu().numpy()[0]
    
    # RoBERTa inference
    inputs_rob = roberta_tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
    with torch.no_grad():
        out_rob = roberta_model(**inputs_rob)
        rob_probs = torch.softmax(out_rob.logits, dim=-1).cpu().numpy()[0]
    
    # Weighted ensemble
    ensemble_probs = 0.7 * deb_probs.astype(float) + 0.3 * rob_probs.astype(float)
    
    # Get top-3 predictions
    order = ensemble_probs.argsort()[::-1]
    top3 = [labels[i] for i in order[:3]]
    top3_str = " ".join(top3)
    
    submission_data.append({"id": row["id"], "prediction": top3_str})

submission_df = pd.DataFrame(submission_data)
submission_df.to_csv("submission.csv", index=False)

print(f"Number of prediction rows: {len(submission_df)}")

Number of prediction rows: 500


### Question 6:

For the first 50 rows of test.csv, create two versions of every prompt:

1.Original prompt

2.Instruction-augmented prompt by prepending: "Answer the following multiple-choice question carefully:"

Run inference using DeBERTa on both versions.

Average the predicted probabilities from both passes.

Question 6:

How many of the first 50 rows produce a different Top-1 prediction after applying Test-Time Augmentation?


In [9]:
import torch

n_changed = 0

for _, row in test.head(50).iterrows():
    orig = row["prompt"]
    aug = "Answer the following multiple-choice question carefully: " + orig

    inputs_orig = deberta_tokenizer(orig, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
    inputs_aug = deberta_tokenizer(aug, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)

    with torch.no_grad():
        out_orig = deberta_model(**inputs_orig)
        out_aug = deberta_model(**inputs_aug)

    probs_orig = torch.softmax(out_orig.logits, dim=-1).cpu().numpy()[0]
    probs_aug = torch.softmax(out_aug.logits, dim=-1).cpu().numpy()[0]

    avg_probs_tta = (probs_orig + probs_aug) / 2.0

    if probs_orig.argmax() != avg_probs_tta.argmax():
        n_changed += 1

print(n_changed)

0


### Question 7:

Process the first 100 rows of test.csv. And compare the Top-1 prediction from:

1. DeBERTa

2. Weighted Ensemble

Question 7:

How many rows have different Top-1 predictions?


In [10]:
count_diff = 0

for _, row in test.head(100).iterrows():
    text = row["prompt"]
    inputs_deb = deberta_tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
    inputs_rob = roberta_tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)

    with torch.no_grad():
        out_deb = deberta_model(**inputs_deb)
        out_rob = roberta_model(**inputs_rob)

    deb_probs = torch.softmax(out_deb.logits, dim=-1).cpu().numpy()[0]
    rob_probs = torch.softmax(out_rob.logits, dim=-1).cpu().numpy()[0]

    ensemble_probs = 0.7 * deb_probs.astype(float) + 0.3 * rob_probs.astype(float)

    if deb_probs.argmax() != ensemble_probs.argmax():
        count_diff += 1

print(count_diff)

0


### Question 8:

For the first 100 rows of test.csv, record the highest class probability (confidence) predicted by:

1. DeBERTa

2. Weighted Ensemble

For every row, compute:

Confidence Gain = Ensemble Confidence−DeBERTa Confidence

Question 8:

How many rows have a positive confidence gain (greater than 0)?


In [11]:
import torch

positive_gain_count = 0

for _, row in test.head(100).iterrows():
    text = row["prompt"]

    # DeBERTa inference
    inputs_deb = deberta_tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
    with torch.no_grad():
        out_deb = deberta_model(**inputs_deb)
    deb_probs = torch.softmax(out_deb.logits, dim=-1).cpu().numpy()[0]
    deb_conf = deb_probs.max()

    # RoBERTa inference
    inputs_rob = roberta_tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
    with torch.no_grad():
        out_rob = roberta_model(**inputs_rob)
    rob_probs = torch.softmax(out_rob.logits, dim=-1).cpu().numpy()[0]

    # Weighted ensemble and confidence gain
    ensemble_conf = (0.7 * deb_probs + 0.3 * rob_probs).max()
    if ensemble_conf > deb_conf:
        positive_gain_count += 1

print(positive_gain_count)

0


### Question 9:

For the first 100 rows of test.csv, compare the Top-3 prediction strings generated by:

1. DeBERTa alone

2. Weighted Ensemble

Question 9:

How many rows have at least one change in their ordered Top-3 ranking after ensembling?


Examples:

A C D vs. A D C



In [12]:
diff_count = 0

for _, row in test.head(100).iterrows():
    text = row["prompt"]
    
    # DeBERTa inference
    inputs_deb = deberta_tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
    with torch.no_grad():
        out_deb = deberta_model(**inputs_deb)
    deb_probs = torch.softmax(out_deb.logits, dim=-1).cpu().numpy()[0]
    
    # RoBERTa inference
    inputs_rob = roberta_tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
    with torch.no_grad():
        out_rob = roberta_model(**inputs_rob)
    rob_probs = torch.softmax(out_rob.logits, dim=-1).cpu().numpy()[0]
    
    # DeBERTa Top-3
    deb_order = deb_probs.argsort()[::-1]
    deb_top3 = " ".join([labels[i] for i in deb_order[:3]])
    
    # Weighted ensemble Top-3
    ensemble_probs = 0.7 * deb_probs.astype(float) + 0.3 * rob_probs.astype(float)
    ens_order = ensemble_probs.argsort()[::-1]
    ens_top3 = " ".join([labels[i] for i in ens_order[:3]])
    
    # Compare
    if deb_top3 != ens_top3:
        diff_count += 1

print(diff_count)

0


### Question 10:

Using the Top-3 predictions generated by your weighted ensemble for the first 100 validation samples, compute the MAP@3 score.

Question 10:


What is the final MAP@3 score? 

(Round to 4 decimal places.)

In [13]:
def mapk_3(y_true, y_pred):
    score = 0.0
    for yt, yp in zip(y_true, y_pred):
        try:
            rank = yp.index(yt)
            score += 1.0 / (rank + 1)
        except ValueError:
            pass
    return score / len(y_true)

y_true = []
y_pred = []

for _, row in train.head(100).iterrows():
    text = row["prompt"]

    inputs_deb = deberta_tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
    inputs_rob = roberta_tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)

    with torch.no_grad():
        out_deb = deberta_model(**inputs_deb)
        out_rob = roberta_model(**inputs_rob)

    deb_probs = torch.softmax(out_deb.logits, dim=-1).cpu().numpy()[0]
    rob_probs = torch.softmax(out_rob.logits, dim=-1).cpu().numpy()[0]

    ensemble_probs = 0.7 * deb_probs.astype(float) + 0.3 * rob_probs.astype(float)
    top3 = [labels[i] for i in ensemble_probs.argsort()[::-1][:3]]

    y_true.append(row["answer"])
    y_pred.append(top3)

score = mapk_3(y_true, y_pred)
print(f"{score:.4f}")

0.3733
